# Gold: lab_test_trends

Builds `adb_medsync.gold.lab_test_trends` per the Silver-to-Gold STTM: one row per test type per calendar month with abnormal/critical rates and a month-over-month abnormal-rate trend (LAG). Driven by `silver.lab_results`, LEFT JOINed to `silver.calendar` for month and quarter names. Full rebuild.

In [0]:
%sql
-- Target: adb_medsync.gold.lab_test_trends | Grain: one row per test_name per calendar month | Load: FULL
CREATE OR REPLACE TABLE adb_medsync.gold.lab_test_trends
COMMENT 'Gold: one row per lab test type per month with abnormal/critical rates and month-over-month trend (STTM silver-to-gold)'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                                    AS year,
    month(test_date)                                   AS month,
    CAST(date_trunc('MONTH', test_date) AS DATE)       AS month_start,
    COUNT(*)                                           AS total_tests,
    COUNT_IF(result_status = 'Abnormal')               AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')               AS critical_test_count
  FROM adb_medsync.silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date), CAST(date_trunc('MONTH', test_date) AS DATE)
),
rates AS (
  SELECT
    *,
    CAST(abnormal_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(critical_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly
),
with_lag AS (
  SELECT
    *,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates
)
SELECT
  r.test_name                                                          AS test_name,
  CAST(r.year AS INT)                                                  AS year,
  CAST(r.month AS INT)                                                 AS month,
  c.month_name                                                         AS month_name,
  c.quarter_name                                                       AS quarter_name,
  CAST(r.total_tests AS INT)                                           AS total_tests,
  CAST(r.abnormal_test_count AS INT)                                   AS abnormal_test_count,
  CAST(r.critical_test_count AS INT)                                   AS critical_test_count,
  r.abnormal_rate_pct                                                  AS abnormal_rate_pct,
  r.critical_rate_pct                                                  AS critical_rate_pct,
  CAST(r.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                 AS prev_month_abnormal_rate_pct,
  CAST(r.abnormal_rate_pct - r.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_lag r
LEFT JOIN adb_medsync.silver.calendar c
  ON c.date_key = CAST(date_format(r.month_start, 'yyyyMMdd') AS INT)

In [0]:
target_table = "adb_medsync.gold.lab_test_trends"
updated_df = spark.read.table(target_table)
record_count = updated_df.count()

print(f"Records loaded into {target_table}: {record_count}")
dbutils.notebook.exit(str(record_count))